# LDDMM Femur Statistical Shape Model - Google Colab Training

This notebook trains an LDDMM-based statistical shape model for femur bones using Google Colab's GPU resources.

1. **Atlas Building**: Computes the Fréchet mean (average shape) of all training femurs using Large Deformation Diffeomorphic Metric Mapping (LDDMM)
2. **Tangent PCA**: Performs Principal Component Analysis in the tangent space at the atlas to capture the main modes of shape variation

This requires a Google account (for Drive storage) and GPU resources.

After training, you'll have a statistical shape model that can:
- Generate new plausible femur shapes
- Interpolate between existing shapes  
- Quantify principal modes of variation in the population



In [ ]:
# =============================================================================
# STEP 1: Mount Google Drive
# =============================================================================
# This allows us to save the trained model to Google Drive for persistent storage.
# You'll be prompted to authorize access to your Drive.

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# STEP 2: Clone the Femur_Modeling Repository
# =============================================================================

!git clone https://github.com/Basile-Mouret/Femur_Modeling.git
%cd Femur_Modeling

Cloning into 'Femur_Modeling'...
remote: Enumerating objects: 3821, done.
remote: Counting objects: 100% (238/238), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 3821 (delta 108), reused 150 (delta 71), pack-reused 3583 (from 4)
Receiving objects: 100% (3821/3821), 331.80 MiB | 16.45 MiB/s, done.
Resolving deltas: 100% (1296/1296), done.
Updating files: 100% (2161/2161), done.


In [ ]:
# =============================================================================
# STEP 3: Install Dependencies
# =============================================================================
# Installs all required Python packages:
#   - scikit-shapes: LDDMM registration and shape analysis
#   - numpy, scipy: Numerical computation
#   - torch: GPU-accelerated tensor operations (used by scikit-shapes)
#   - pyvista: 3D mesh visualization (optional, for debugging)
#
# NOTE: After installation, the runtime may need to restart. Run this cell
#       again if you get import errors.

!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 41.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 19.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 MB 16.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 152.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.4/872.4 kB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.4/740.4 kB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 42.0 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.4/282.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
# =============================================================================
# STEP 4: Train the LDDMM Statistical Shape Model
# =============================================================================
# Output files saved to models/lddmm_pca/:
#   - atlas.npz: Atlas shape vertices, faces, and registration momenta
#   - tangent_pca.npz: PCA components, eigenvalues, and mean tangent vector
#   - model_info.json: Training configuration and metadata
#
# ═══════════════════════════════════════════════════════════════════════════
# Available arguments (edit the command below to customize):
#
#   --config PRESET       LDDMM preset: "for_femurs" (default), "high_precision", "fast"
#   --n-components N      Number of PCA components to keep (default: all)
#   --scale SIGMA         Kernel bandwidth in mm (default: ~15)
#   --n-steps N           ODE integration steps (default: 5)
#   --n-iter N            L-BFGS iterations per registration (default: 100)
#   --regularization W    Deformation energy weight λ (default: 0.01)
#   --kernel TYPE         "gaussian" (default) or "cauchy"
#   --quiet               Less verbose output
#
# Examples:
#   !python -m lddmm.femur_lddmm.femur_model --config fast
#   !python -m lddmm.femur_lddmm.femur_model --n-components 10 --scale 12 --n-iter 150
# ═══════════════════════════════════════════════════════════════════════════

!python -m lddmm.femur_lddmm.femur_model

Femur Tangent PCA Model Builder

Data directories: ['/content/Femur_Modeling/data/training', '/content/Femur_Modeling/data/validation']
Output directory: /content/Femur_Modeling/models/lddmm_pca

[1/4] Loading femur data...
[FemurDataLoader] Found 22 files in /content/Femur_Modeling/data/training
  Loaded L_Femur_11_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_13_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_14_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_15_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_16_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_19_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_20_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_21_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded L_Femur_23_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded R_Femur_01_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded R_Femur_02_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded R_Femur_03_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded R_Femur_04_DECIM.obj.FINAL.obj: (18291, 3)
  Loaded R_Femur_05_DECIM.

In [ ]:
# =============================================================================
# STEP 5: Save Model to Google Drive
# =============================================================================
# Files copied:
#   - atlas.npz (~2-5 MB): Contains the atlas shape and registration momenta
#   - tangent_pca.npz (~1-2 MB): Contains PCA model for shape statistics
#   - model_info.json (~1 KB): Human-readable training configuration
#
# After this step, you can download the model from Drive or use it in
# other notebooks by mounting Drive and loading from this path.

!mkdir -p "/content/drive/MyDrive/Femur_Modeling/models"
!cp -r models/lddmm_pca "/content/drive/MyDrive/Femur_Modeling/models/"
print("✓ Model saved to Google Drive: MyDrive/Femur_Modeling/models/lddmm_pca/")